In [13]:
%matplotlib inline

from sklearn.preprocessing import StandardScaler, MinMaxScaler, PolynomialFeatures
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report, f1_score, precision_score, recall_score
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import RFECV

from sklearn.linear_model import RidgeClassifierCV
from sklearn.linear_model import LinearRegression

from ultralytics import YOLO
from PIL import Image

import matplotlib.pyplot as plt
import mediapipe as mp
import cv2
import time
import numpy as np
import pandas as pd
import os

import warnings
warnings.filterwarnings('ignore')

In [14]:
# Загрузка модели
model = YOLO('yolo11n-pose.pt')  # load an official model

points = 17
path_train = "DATASET/TRAIN/" # Путь к датасету
path_test = "DATASET/TEST/" # Путь к датасету

In [15]:
Key_Points_YOLO = ['Nose', 'LeftEye', 'RightEye', 'LeftEar', 'RightEar', 
                   'LeftShoulder', 'RightShoulder', 'LeftElbow', 'RightElbow', 
                   'LeftWrist', 'RightWrist', 'LeftHip', 'RightHip', 
                   'LeftKnee', 'RightKnee', 'LeftAnkle', 'RightAnkle'
                  ]
classes = ['downdog', 'goddess', 'plank', 'tree', 'warrior2']

# Создаем словарь для замены
mapping = {'downdog': 0, 'goddess': 1, 'plank': 2, 'tree': 3, 'warrior2': 4}


In [16]:
# Загрузка и создание обучающего набора
data_train = pd.read_csv("dataset_train_yolo.csv")
# Заменяем значения
data_train['target'] = data_train['target'].map(mapping)

X_yolo = data_train.iloc[:, 2:-1]
Y_yolo = data_train['target']

# Загрузка и создание тестового набора
data_test = pd.read_csv("dataset_test_yolo.csv")
# Заменяем значения
data_test['target'] = data_test['target'].map(mapping)

X_test_yolo = data_test.iloc[:, 2:-1]
Y_test_yolo = data_test['target']

# Задаём гиперпараметры
params = {
    'n_estimators': 100,      # Количество деревьев
    'max_depth': 10,           # Максимальная глубина дерева
    'min_samples_split': 5,   # Минимальное число образцов для разделения узла
    'min_samples_leaf': 1,    # Минимальное число образцов в листе
    'random_state': 42,       # Для воспроизводимости
}

# Создаём модель
best_model = RandomForestClassifier(**params)

# Обучаем модель
best_model.fit(X_yolo, Y_yolo)

path = path_test
image_err = []
time_inference = 0
time_proc = 0
time_cl = 0
for dr in os.listdir(path): # Перебор папок с видами поз
    for image in os.listdir(path + "/" +dr): # Перебор файлов в каждой папке
        start_file = time.time()
        name_file = path + "/" + dr + "/" + image
        start_time = time.time()
        #data = preprocessing(name_file)
                
        # Загрузка выбранного файла
        #path ='DATASET/Train/goddess/00000137.jpg'# 'DATASET/Test/plank/00000015.jpg'             #'DATASET/Test/warrior2/00000093.jpg' 'DATASET/Train/goddess/00000137.jpg'
        temp = []
        img = cv2.imread(name_file)
        # Копирование и конвертация изображения в RGB
        imageWidth, imageHeight = img.shape[:2]
        imgRGB = cv2.cvtColor(img, cv2.COLOR_BGR2RGB) # Преобразование BGR модели OpenCV в RGB модель, с которой работает YOLO
        
        # Обнаружение позы            
        results = model(imgRGB)  # predict on an image 
        
        # Построение скелетной модели
        # Извлечение результатов
        for result in results[0]:
            xy = result.keypoints.xy  # x and y coordinates
            xyn = result.keypoints.xyn  # normalized
            kpts = result.keypoints.data  # x, y, visibility (if available)
        keypoints = np.array(xy[0])
        normal_keypoints = np.array(xyn[0])
        for i in range(len(normal_keypoints)):
            temp = temp + [normal_keypoints[i][0], normal_keypoints[i][1]] # Добавление ключевых точек
            
        print('Время препроцессинга: ', time.time()-start_time)
        time_proc += time.time()-start_time
        
        # Предсказание построенным классификатором класса позы по построенной скелетной модели    
        start_time = time.time()
        res = best_model.predict([temp])
        print('Время определения класса: ', time.time()-start_time) 
        time_cl += time.time()-start_time
        
        label_predict = res[0]

        # Вывод результата
        print('Эталонное название позы - ', name_file.split('/')[3]) 
        print('Предсказанное название позы - ', classes[label_predict]) 
        
        time_vis = time.time()
        time_file = time_vis - start_file

        
        if name_file.split('/')[3] != classes[label_predict]:
            image_err.append(name_file)

        # Визуализация результата
        for i, r in enumerate(results):
            # Plot results image
            im_bgr = r.plot(conf = False, kpt_radius = 7, boxes = False, masks = False)  # BGR-order numpy array
            im_rgb = Image.fromarray(im_bgr[..., ::-1])  # RGB-order PIL image
            #r.show()
            r.save(filename='ERR/results/'+dr+image)
            if name_file.split('/')[3] != classes[label_predict]:
                r.save(filename='ERR/'+dr+'_'+str(classes[label_predict])+'_'+image)
        
        
        time_inference += time_file
    
print(image_err) 
    
print(time_inference)
print(time_proc/470)
print(time_cl/470)


0: 320x640 1 person, 69.6ms
Speed: 1.3ms preprocess, 69.6ms inference, 0.8ms postprocess per image at shape (1, 3, 320, 640)
Время препроцессинга:  0.15113592147827148
Время определения класса:  0.006076812744140625
Эталонное название позы -  goddess
Предсказанное название позы -  tree

0: 640x640 1 person, 97.6ms
Speed: 2.9ms preprocess, 97.6ms inference, 0.8ms postprocess per image at shape (1, 3, 640, 640)
Время препроцессинга:  0.12311387062072754
Время определения класса:  0.005928754806518555
Эталонное название позы -  goddess
Предсказанное название позы -  goddess

0: 640x640 1 person, 74.6ms
Speed: 2.6ms preprocess, 74.6ms inference, 0.9ms postprocess per image at shape (1, 3, 640, 640)
Время препроцессинга:  0.09145641326904297
Время определения класса:  0.006272315979003906
Эталонное название позы -  goddess
Предсказанное название позы -  goddess

0: 480x640 1 person, 60.9ms
Speed: 1.7ms preprocess, 60.9ms inference, 0.9ms postprocess per image at shape (1, 3, 480, 640)
Врем

libpng warning: iCCP: known incorrect sRGB profile


0: 384x640 1 person, 50.0ms
Speed: 1.5ms preprocess, 50.0ms inference, 0.8ms postprocess per image at shape (1, 3, 384, 640)
Время препроцессинга:  0.05624532699584961
Время определения класса:  0.0061261653900146484
Эталонное название позы -  goddess
Предсказанное название позы -  goddess

0: 512x640 1 person, 84.9ms
Speed: 1.9ms preprocess, 84.9ms inference, 1.2ms postprocess per image at shape (1, 3, 512, 640)
Время препроцессинга:  0.0973656177520752
Время определения класса:  0.005794048309326172
Эталонное название позы -  goddess
Предсказанное название позы -  goddess

0: 640x640 1 person, 70.0ms
Speed: 3.3ms preprocess, 70.0ms inference, 0.8ms postprocess per image at shape (1, 3, 640, 640)
Время препроцессинга:  0.10453224182128906
Время определения класса:  0.0060198307037353516
Эталонное название позы -  goddess
Предсказанное название позы -  warrior2

0: 640x608 1 person, 94.0ms
Speed: 2.4ms preprocess, 94.0ms inference, 0.9ms postprocess per image at shape (1, 3, 640, 608)


libpng warning: iCCP: known incorrect sRGB profile



0: 480x640 1 person, 92.7ms
Speed: 2.2ms preprocess, 92.7ms inference, 0.9ms postprocess per image at shape (1, 3, 480, 640)
Время препроцессинга:  0.4380779266357422
Время определения класса:  0.0070171356201171875
Эталонное название позы -  tree
Предсказанное название позы -  tree

0: 640x480 1 person, 60.6ms
Speed: 1.9ms preprocess, 60.6ms inference, 0.8ms postprocess per image at shape (1, 3, 640, 480)
Время препроцессинга:  0.06824994087219238
Время определения класса:  0.006375789642333984
Эталонное название позы -  tree
Предсказанное название позы -  tree

0: 640x544 1 person, 106.1ms
Speed: 1.9ms preprocess, 106.1ms inference, 0.8ms postprocess per image at shape (1, 3, 640, 544)
Время препроцессинга:  0.11283135414123535
Время определения класса:  0.0073659420013427734
Эталонное название позы -  tree
Предсказанное название позы -  tree

0: 448x640 1 person, 60.5ms
Speed: 2.1ms preprocess, 60.5ms inference, 0.9ms postprocess per image at shape (1, 3, 448, 640)
Время препроцесс

libpng warning: iCCP: known incorrect sRGB profile



0: 480x640 1 person, 62.7ms
Speed: 2.7ms preprocess, 62.7ms inference, 0.7ms postprocess per image at shape (1, 3, 480, 640)
Время препроцессинга:  0.4061293601989746
Время определения класса:  0.00649714469909668
Эталонное название позы -  tree
Предсказанное название позы -  tree

0: 640x416 1 person, 71.6ms
Speed: 1.9ms preprocess, 71.6ms inference, 0.7ms postprocess per image at shape (1, 3, 640, 416)
Время препроцессинга:  0.10921955108642578
Время определения класса:  0.006310701370239258
Эталонное название позы -  tree
Предсказанное название позы -  warrior2

0: 448x640 1 person, 58.7ms
Speed: 1.8ms preprocess, 58.7ms inference, 0.9ms postprocess per image at shape (1, 3, 448, 640)
Время препроцессинга:  0.06915974617004395
Время определения класса:  0.007330417633056641
Эталонное название позы -  tree
Предсказанное название позы -  tree

0: 640x448 1 person, 60.0ms
Speed: 2.0ms preprocess, 60.0ms inference, 0.8ms postprocess per image at shape (1, 3, 640, 448)
Время препроцесси